# 🌧️ Analisis Curah Hujan Multi-Skala: Satelit Kebumen vs AWS IoT Jerukagung
### 📍 Evaluasi Presipitasi Resolusi Per Jam (*Hourly*) & Resolusi Per Hari (*Daily*) Menggunakan `Data_Curah_Hujan_Kebumen.csv`

---
### 📌 Ringkasan Eksekutif
Notebook ini membandingkan data presipitasi satelit & reanalisis pada dua domain waktu:
1. **Data Harian**: Menggunakan dataset `Data_Curah_Hujan_Kebumen.csv` (8 produk: `CHIRPS_RNL`, `CHIRPS_SAT`, `CHIRPS_FNL`, `GSMaP`, `IMERG`, `PERSIANN`, `ERA5`, `ERA5_LAND`) vs agregasi harian AWS IoT Jerukagung.
2. **Data Per Jam**: Menggunakan dataset resolusi 1-jam sinkron (`id-05_clear_data_hourly.csv`, GSMaP, IMERG, ERA5 Hourly).


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 150

print("Library dan environment berhasil dimuat.")


## 📂 1. Pemuatan Dataset Harian (8 Satelit Kebumen) & Jam-jaman AWS IoT

In [ ]:
data_dir = r'../Data_Satelit'
if not os.path.exists(data_dir):
    data_dir = r'd:/Github/Projek_Rainfall/Google_Earth_Engine/Data_Satelit'

# 1. Dataset Harian Kebumen
df_kebumen = pd.read_csv(os.path.join(data_dir, 'Data_Curah_Hujan_Kebumen.csv'))
df_kebumen['Date'] = pd.to_datetime(df_kebumen['datetime_utc'] if 'datetime_utc' in df_kebumen.columns else df_kebumen['Date'])
sat_cols = ['CHIRPS_RNL', 'CHIRPS_SAT', 'CHIRPS_FNL', 'GSMaP', 'IMERG', 'PERSIANN', 'ERA5', 'ERA5_LAND']
df_daily_sat = df_kebumen.set_index('Date')[sat_cols]

# 2. Dataset Jam-jaman AWS IoT
df_aws = pd.read_csv(os.path.join(data_dir, 'id-05_clear_data_hourly.csv'))
df_aws['Date'] = pd.to_datetime(df_aws['datetime_utc'])
df_aws_daily = pd.DataFrame()
df_aws_daily['rain_aws'] = df_aws.set_index('Date')['rainrate'].resample('D').apply(lambda s: s.sum(min_count=20))

# Gabung Data Harian Master
df_daily_master = df_daily_sat.join(df_aws_daily, how='inner').dropna(subset=['rain_aws'])

print(f"Total Hari Valid Overlap (8 Satelit Kebumen vs AWS IoT Harian): {len(df_daily_master):,} hari")
display(df_daily_master.head())


## 📊 2. Ringkasan Metrik Evaluasi: Per Jam vs Per Hari

In [ ]:
df_eval_d = pd.read_csv(r'Hasil_Analisis/ringkasan_evaluasi_harian_8satelit_vs_aws.csv')
print("=== EVALUASI 8 PRODUK SATELIT HARIAN VS AWS IOT ===")
display(df_eval_d)

df_multi = pd.read_csv(r'Hasil_Analisis/ringkasan_multi_skala_jam_vs_hari.csv')
print("=== PERBANDINGAN MULTI-SKALA (JAM VS HARI) ===")
display(df_multi)


## 🖼️ 3. Visualisasi Hasil Analisis Multi-Skala

In [ ]:
from IPython.display import Image, display

plots = [
    '01_bar_evaluasi_8satelit_vs_aws_harian.png',
    '02_scatter_hexbin_8satelit_vs_aws_harian.png',
    '03_perbandingan_scatter_jam_vs_hari.png',
    '04_bar_lonjakan_akurasi_jam_vs_hari.png',
    '05_siklus_diurnal_24jam_cuaca_hujan.png',
    '06_skor_kontingensi_deteksi_hujan_harian.png',
    '07_kurva_massa_ganda_harian.png',
    '08_heatmap_korelasi_harian_semua_produk.png'
]

for p in plots:
    fp = os.path.join('Hasil_Analisis', p)
    if os.path.exists(fp):
        print(f"=== {p} ===")
        display(Image(fp))


## 🎯 4. Kesimpulan & Rekomendasi
1. **NASA GPM IMERG** merupakan produk presipitasi harian terbaik terhadap observasi darat AWS IoT di Kebumen ($r = 0.619, ho = 0.642$).
2. Agregasi harian menghasilkan lonjakan korelasi sebesar $+49.5\%$ dibandingkan resolusi 1-jam.
3. Hujan konvektif sore hari (15:00–18:00 WIB) mendominasi kejadian hujan di Stasiun Jerukagung Kebumen.
